# JHDN7CF1C03X5

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
import math
import os
import gc
from pathlib import Path
import re
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Set directory to project root
def find_project_root(start: Path = Path().absolute()) -> Path:
    for parent in start.parents:
        if (parent / "requirements.txt").exists(): return parent
    return start 
os.chdir(find_project_root())

# Custom packages
from tools.labeling_functions import clean_and_relabel_restaurant, plot_dish_time_series
from tools.coverage_functions import plot_time_series

# Preemptively set new Pandas option, also set matplotlib to close
pd.options.mode.copy_on_write = True
%matplotlib inline
%config InlineBackend.close_figures=True

# Allow reloading of custom Python classes without resetting kernel
pd.set_option('display.max_rows', 100)
%load_ext autoreload
%autoreload 2

Read data from parquet files

In [ ]:
# Load formatted data
%store -r static_data_merged
%store -r sales_data_merged

# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"data/2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"data/2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    
    %store static_data_merged
    
# Data already exists
else:
    static_data = static_data_merged.copy()

# 

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"data/2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"data/2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
        
    %store sales_data_merged
    
# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()

# 

# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

# True promos
%store -r before_after_details_true
if 'before_after_details_true' not in locals():
    before_after_details_true = pd.read_csv('data/before_after_details_true.csv', index_col='location_id')
    %store before_after_details_true
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']

# Timezones
%store -r timezones
if 'timezones' not in locals():
    timezones = pd.read_csv('data/timezones.csv', index_col='location_id')['timezone'].to_dict()
    for loc_id, df in sales_and_menu_data.items():
        df.index = df.index.tz_convert(timezones[loc_id])
        sales_and_menu_data[loc_id] = df
    %store timezones

%store -r restaurants_by_4m_coverage
if 'restaurants_by_4m_coverage' not in locals():
    restaurants_by_4m_coverage = pd.read_csv('data/2_palate_data_parquet_cleaned/restaurants_by_4m_coverage.csv')['location_id'].tolist()
    %store restaurants_by_4m_coverage

loc_id = 'JHDN7CF1C03X5'
df_uncleaned = sales_and_menu_data[loc_id]

locations = list(sales_and_menu_data.keys())
for other_loc_id in locations:
    if other_loc_id != loc_id:
        del sales_and_menu_data[other_loc_id]
        del sales_data_merged[other_loc_id]
gc.collect()

In [ ]:
print(df_uncleaned.query('item_name.str.contains("Beyond")')['item_quantity'].sum())
print(df_uncleaned.loc[promo_date:promo_date+pd.DateOffset(days=60)].query('item_name.str.contains("Beyond")')['item_quantity'].sum())

In [ ]:
# creating food_df to exclude drinks
beverages = ["Coffee & Tea",
             "Water",
             "Juice",
             "Sports & Health Drink",
             "Soda"
]

food_df = (df
           .query('~dish_category.isin(@beverages)')
)

# Regex
exclusion_phrases = [
    "No", 
    "Remove", 
    "Instead Of", 
    "Or", 
    "Un", 
    "Vegan", 
    "Vegsan", 
    "Vergan", 
    ]
#general_phrases_to_exclude = ["",]

contains_cheese_words = [
    "No Chee", "No. Ch", "Or Che", "No Moz",
     # meet this literally so regex = False
    "No Feta", ", Or Feta",
    "No Parm", "No Parnm",
    "Extra Chee",
]
contains_cheese = '|'.join(map(re.escape, contains_cheese_words))
remove_yogurt_words = [
    "No Yo", "No Greek", "Take Off Greek Yogurt",
    "Or Greek",
    "No Frozen Yogurt", "No Fro", "No Fr Yo", "No Fy", "Or Fr",
]
remove_yogurt = '|'.join(map(re.escape, remove_yogurt_words))


# if for a given item_name, the item_modifications contains remove_cheese or remove_yogurt then make the item_name to list, and add this list to dairy_items

contains_dairy_mask = (
    (food_df["item_modifications"].str.contains(remove_yogurt, na=False, regex=False))|
    (food_df["item_modifications"].str.contains(contains_cheese, na=False, regex=False))
)
items_containing_dairy = food_df.loc[contains_dairy_mask, 'item_name'].unique().tolist()
dairy_products = [
    "Yogurt", "Whey", 
    "Froyo", "Greek",
    #"Che", "Chhese", #,check exclusion phrases for non-cheese words
    "Cheese", "Chhese", "Chz", 
    "Cheddar", "Cheedar",
    "Sour Cream",
    "Moz",
    "Ranch",
    "Parm", "Parnm", 
    ]
dairy_regex_words = '|'.join(dairy_products)
dairy_exclusion_phrases = [
    "No Dairy", "No Ranch",
    "No Yo", 
    "No Frozen Yogurt", "No Fro", "No Fr Yo", "No Fy",#Checked all strings in item_mod containing "No Fro", only no frozen yogurt pops up
    "No Greek", "Take Off Greek Yogurt", "No Quinoa Or Greek Yogurt",
    #"Vegan Cheese Instead Of Chedder",
    "Plant Protein Instead Of Whey", "Plant Instead Of Whey",
    "Regular  Vegan",
    "No Whey", "Plant Not Whey",
    "No Feta",
    "Vegan Cheese", "Vegsan Cheese", "Vergan Cheese", "Vegan Chz",
    "No Cheese", "No Mushroom Or Cheese", "No Chz", "No Cheese,No", "No Cheese!", "Nom Cheese",
    "No Cheddar", "No Bacon, Cheddar", 
    "Cher", "Cheyenne",  #non-cheese words
    
    "Nom Cheese"
    ]

dairy_exclusion_regex = '|'.join(dairy_exclusion_phrases)
#specific_dairy_exclusion_regex = '|'.join(dairy_exclusion_phrases)
dairy_removal_regex = rf"(?<=\b(?:{exclusion_phrases})\s)(?:{dairy_regex_words})"
#dairy_exclusion_regex = rf"(?:{specific_dairy_exclusion_regex}|{dairy_removal_regex})"


dairy_regex = rf"^(?:{dairy_regex_words})|(?<!\b(?:{exclusion_phrases})\s)(?:{dairy_regex_words})"
dairy_dish = [
    'Chunky Monkey', 'Pineapple Paradise', 'Avocado Dream', 'Protein Power', 'Yogurt',
    #"Mediterranean Spinach Salad", 
    "Portobello Balsamic Toast",
    "Pumpkin Spice Smoothie", "Greek Yogurt Bowl", 
    #"$3 Combo",
    #"Healthy Start Panini", "Spicy Chicken Panini", "Fiesta Bowl", "Mission Burrito Wrap",
    #"Cali Bowl", 
    "Kid'S Grilled Cheese Quesadilla", "Cheese"
    "Cali Bowl",
    "Fiesta Bowl",
]
dairy_dish_regex = '|'.join(dairy_dish)


# intersection of meat and dairy exclusion = 

meat_products_in_plantbased = [
    'Chicken', 'Cjicken', 'Chicen', 'Chickjen', 'Chicke', 'Chicjen', 'Chkn',
    # Where food_df.query('item_modifications.str.contains("Bacon")'), all removing bacon denoted by "No Bacon"
    "Bacon", 'Turkey', "Turkey Bacon", "Bacon,",
    "Tuna",
    "Extra Meat"
    # item_mod contains "Meat"
    
                               ]
meat_regex_words = '|'.join(meat_products_in_plantbased)
meat_regex = rf"(?<!\b(?:{exclusion_phrases})\s)(?:{meat_regex_words})"
meat_substitute_phrases = [
    "Un'Chicken", "Veg Chicken", "Unchicken", "Un Chiken", "Plant Chicken","Un Chicken", "Unchuicken",
    "Vegan", "Beyond", "No Chicken", "No Tb"
    ]
meat_exclusion_phrases = [
    "No Chicken", 
    "No Tb", 
    "No Bacon", 
    "No Tuna", 
    "Un'Chicken", 
    "Veg Chicken", 
    "Unchicken", 
    "Un Chiken", 
    "Plant Chicken",
    "Un Chicken", 
    "Unchuicken", 
    "Regular  Vegan",]
meat_exclusion_regex = '|'.join(meat_substitute_phrases)
# meat_removal_regex = rf"(?<=\b(?:{exclusion_phrases})\s)(?:{meat_regex_words})"
# meat_exclusion_regex = rf"(?:{meat_substitute_regex}|{meat_removal_regex})"

# did not get rid of "Regular Add Chicken And No Cheese Or Greek Yogurt" because it has exclusion phrase 'Che'?

meat_dish = [
    'Chicken', 
    "Turkey Bacon", "Turkey", 
    "Breakfast Panini", # On website has turkey + bacon and egg, beyond sub, unchicken sub, vegan cheese sub 
    "Santa Fe Quesadilla",
    "Mediterranean Bowl", #has turkey on web-site
    "Mediterranean",
    'Quesadilla Chicken Fiesta',
    'Southewestern Fiesta Wrap', "Albacore Melt Panini",
    'Fiesta Bowl',
    'Breakfast Wrap', 
    'Mission Burrito Wrap',
    #"$3 Combo",
    "Healthy Start Panini",
    "Teriyaki Bowl",
    "Early Bird Wrap",
    "Tex Bowl",
    "Turkey Bacon Avocado Panini",
    "Kid'S Grilled Cheese Quesadilla",
]

meat_dish_regex = '|'.join(meat_dish)
egg = ["Egg", "Breakfast Wrap", ]
no_egg = ["No Egg", "Regular  Vegan",]
egg_regex = '|'.join(egg)
egg_exclusion_regex = '|'.join(no_egg)
egg_dish = ["Egg", "Rx Bars", "Cali Bowl",]
egg_dish_regex = '|'.join(egg_dish)

# Creating boolean mask and making changes to it
# Create a boolean mask for rows that match the specific conditions
# Now, food_df still contains all rows, but only the matching ones have the updated item_name


meat_mask = (# if name has meat or item_mods have meat
    (
        (food_df["item_name"].str.contains(meat_dish_regex, na=False))|
        (food_df["item_modifications"].str.contains(meat_regex_words, na=False))
    )
    &
    (# and (has meat and asked to remove meat) or (did not ask to remove meat)
        (~food_df["item_modifications"].str.contains(meat_exclusion_regex, na=False))#|
        # (
        #     (food_df["item_modifications"].str.contains(meat_exclusion_regex, na=False))&
        #     (food_df["item_modifications"].str.contains(meat_regex, na=False))
        # )
        
    )
)

dairy_mask = (
    # (food_df["is_plant_based"] == "Yes") & # no need to mention this, to include vegetarian items from both
    (
        (~meat_mask &
            (
                (food_df["item_name"].str.contains(dairy_dish_regex, na=False))|
                (food_df["item_modifications"].str.contains(dairy_regex_words, na=False))
            )
        )&
        (
            (~food_df["item_modifications"].str.contains(dairy_exclusion_regex, na=False))#|
            # (
            #     (food_df["item_modifications"].str.contains(dairy_exclusion_regex, na=False))&
            #     (food_df["item_modifications"].str.contains(dairy_regex, na=False))
            # )
        )
    ) 
)
# if item_modifications.str.contains("No Yo") or ("No Greek") then list that dish in dairy_items


egg_mask = (
    (
        (food_df["item_name"].str.contains(egg_dish_regex, na=False))|
        (food_df["item_modifications"].str.contains(egg_regex, na=False))
    )
    &
    (~food_df["item_modifications"].str.contains(egg_exclusion_regex, na=False))
)

# food_df = food_df.assign(
#     item_name=lambda df: df["item_name"]
#     .mask(meat_mask, "Meat " + df["item_name"])
#     .mask(dairy_mask, "Dairy " + df["item_name"])
#     .mask(egg_mask&~meat_mask, "Egg " + df["item_name"])
# )


meat_items = food_df.loc[meat_mask, 'item_name'].unique().tolist()
vegetarian_items = food_df.loc[dairy_mask|egg_mask, 'item_name'].unique().tolist()

In [ ]:
# Item names to swap based on modications

modification_name_changes = [
    ('Fresh Beyond Burger', 'Bacon', 'Meat'),
    ('Fiesta Bowl', '''Regular No Yogurt, Balsamic Sub Beyond Meat|Regular  No Cheese Sub Beyond Meat|Regular No Greek Yogurt  Sub Beyond Meat|Regular No Yogurt, Extra Salsa Un'Chicken''', 'Vegetarian'),


    ('Kale Caesar Salad','Chicken','Chicken Caesar Salad'),
    ('Kale Caesar Salad','Eggs','Vegetarian Caesar Salad'),
    ("Santa Fe Quesadilla","Beyond|Substitute Veg Chicken","Vegan Santa Fe Quesadilla"), 
    
    ("Protein Power","No Yogurt","Vegan Protein Power"),
    ("Mango Mania","No Fr|No Yog","Vegan Mango Mania"),
    ("Veggie Portobello","No Ch|Vegan","Vegan Portobello"),
    ("Kid'S Smoothie","Chunky Monkey|Protein Power|Pineapple Paradise","Dairy Kid'S Smoothie"),
    ("Avocado Toast","Egg","Egg Avocado Toast"),
    ("Veg Buddha Bowl","No Cheese|Vegan|No F","Vegan Buddha Bowl"),
    ("Kale Vegetarian Wrap","Egg","Egg Vegetarian Wrap"),
    ("$3 Donation Free Child Size Smoothie","Chunky Monkey|Protein Power|Pineapple Paradise","$3 Donation Dairy Free Child Size Smoothie"),
    ("Quinoa & Chickpeas","Eggs","Egg Quinoa & Chickpeas"),
    
    ("$ Combo Meal","Tuna","Meat $ Combo Meal"),
    
    ("Protein Power","No Whey Plant, And No Yogurt","Vegan Protein Power"),
    ("Mission Burrito Wrap","Yog","Vegan Mission Burrito Wrap"),
    ("Turkey Bacon Avocado Panini","Beyond|Un'Chicken|Vegan|Unchicken","Veganized Panini"),
    ("Caesar Blt Chicken Wrap","Che|Vegan","Unchicken Blt Wrap"),
    ("Beach Salad","No Chicken|Un","Chickenless Beach Salad"),
    ("Fiesta Bowl","No","Dairy Fiesta Bowl"),
    ("Mission Burrito Wrap","Che|Yog","Vegan Mission Burrito Wrap"),
    ("Mission Burrito Wrap","Bey|Un","Unchicken/Beyond Mission Burrito Wrap"),
    ("Cali Bowl","Yo|Vegan","Vegan Cali Bowl"),
    ("Caesar Blt Chicken Wrap","Vegan","Vegan Blt Wrap"),
    ("Spicy Chicken Panini","Vegan|Che","Spicy Unchicken Panini"),
    ("Southewestern Fiesta Wrap","Che|Vegan","Vegan Southewestern Fiesta Wrap"),
    ("Bbq Chicken Panini","Che|Veg","Vegan Bbq Unchicken Panini"),
    ("Harvest Signature Bowl","Feta","Vegan Harvest Signature Bowl"),
    ("Mediterranean","Che","Vegan Mediterranean"),
    ("Turkey Pesto Panini","Che","Vegan Pesto Panini"),
    ("Breakfast Wrap","Vegan","Vegan Breakfast Wrap"),
    ("Turkey Bacon Avocado Wrap","Ch|Vegan","Vegan Avocado Wrap"),
    ("Turkey Bacon Avocado Panini","Vegan","Vegan Avocado Panini"),
    
    ("Soup Of The Day","Vegan","Vegan Soup Of The Day"),
    ("$2 Combo","Chic|Chxn","Chicken $2 Combo"),
    ("$ Combo Meal","Plant Protein","Vegan $ Combo Meal"),
    ("$ Combo Meal","Tba|Sesame|Beach Salad|Kale Caesar Salad|Tuna|Turkey|Chicken|Early|Chcnk","Meat $ Combo Meal"),
    ("$ Combo Meal","Panini|Southwestern Wrap|Southwest Wrap|Chunky Monkey|Half And","Vegetarian $ Combo Meal"),
    ("$ Combo Meal","No Ch|Yo|Vegan||","Vegan $ Combo Meal"),
    ("Big Salad Bowls","Protein","Vegan Big Salad Bowls"),
    ("Moroccan Glow Bowl","Che","Vegan Moroccan Glow Bowl"),
    ("Enjoy Life Lentils","Parm","Vegetarian Enjoy Life Lentil Chips"),
    ("Byo Protein/Salad Bowl","Chicken Breast|Turkey","Meat Salad Bowl Chicken/Turkey")
    ]

In [ ]:
# Unique words containing a string 
(pd.Series
(list
(set
(food_df
      #.query('is_plant_based == "Yes"')
      .query('item_modifications.str.contains("Par")')
      ['item_modifications']
      .value_counts()
      .index
      .to_series()
      .reset_index(drop=True)
      .astype(str)
      .str.cat(sep = ' ') #string
      .split() # list
       ))
       , name='item_modifications_yo_word').to_frame()
      .query('item_modifications_yo_word.str.contains("Par")') # can't have strings separated by space, becuse you use it earlier to separate strings
      ['item_modifications_yo_word'].tolist()
      )


In [ ]:
# Is vegan but was labelled meat | reassign is_plantbased == "Yes"
vegan = ['Fresh Beyond Burger',
         'Beyond Burger Combo', 
         'Kale Caesar Salad',
         "Vegan Protein Power",
         "Vegan Mango Mania",
         "Vegan Portobello",
         "Kid'S Smoothie",
         "Avocado Toast",
         "Vegan Buddha Bowl",
         "Kale Vegetarian Wrap",
         "$3 Donation Free Child Size Smoothie",
         "Mediterranean Spinach Salad",
         "Quinoa & Chickpeas",
         "Vegan Protein Power",
         "Abc'S Cookies",
         "Jolly Green",
         "Fiesta Bowl",
         "Vegan Mission Burrito Wrap",
         "$3 Combo",
         "Spicy Chicken Panini",
         "Healthy Start Panini",
         "Tex Bowl",
         "English Egg Muffin",
         "Veganized Panini",
         "Orange",
         'Chips',
         "Chickenless Beach Salad",
         "English Muffin + Coffee",
         "Blue Spirulina",
         "Fiesta Bowl",
         "Vegan Mission Burrito Wrap",
         "Vegan Cali Bowl",
         "Vegan Blt Wrap",
         "Spicy Unchicken Panini",
         "Vegan Southewestern Fiesta Wrap",
         "Chunky Monkey",
         "Vegan Harvest Signature Bowl",
         "Vegan Bbq Unchicken Panini",
         "Vegan Mediterranean",
         "Mandarin Orange Cup",
         "Chicken Caesar Wrap",
         "Teriyaki Bowl",
         "Vegan Pesto Panini",
         "Vegan Breakfast Wrap",
         "Asian Chicken Wrap",
         "Vegan Avocado Wrap",
         "Vegan Avocado Panini",
         "Kids Combo With Squeeze",
         "Kid'S Grilled Cheese Quesadilla",
         "Vegan Soup Of The Day",
         "Cyo (Create Your Own) Squeeze",
         "$2 Combo",
         "Vegan $ Combo Meal",
         "Abc  Cookies",
         "Sabra Snackers", #google search
         "City Chips",#assumption
         "Like Is Good Chips", #assumption
         "Power Bowl Bar",
         "Soba Noodle Bowls", #google search general recipe
         "Vegan Big Salad Bowls",
         "Simply 7 Chips", #assumption cuz chips are vegan
         "Cliff Duos", #google search
         "Bear Granola Bites", #bear naked granola bites google search
         "Vegan Moroccan Glow Bowl",
         "Coco Avocado",
         "English Muffin  +   Squeeze",
         "Enjoy Life Lentils",
         "Protein Bowl",
         "Byo Protein/Salad Bowl"
         ]

# Is vegetarian but was labelled plant-based or meat
vegetarian = ['Beyond Burger With Dairy', 
              'Beyond Burger Combo With Dairy',
              'Protein Power', #has greek and frozen yogurt by default
              'Mango Mania',
              'Dairy Portobello',
              'Egg Avocado Toast',
              'Veg Buddha Bowl', #has feta by default
              'Egg Vegetarian Wrap',
              '$3 Donation Dairy Free Child Size Smoothie',
              'Egg Quinoa & Chickpeas',
              'Protein Power',
              'Yogurt Parfait',
              'Cheesecake',
              'Unchicken Blt Wrap',
              'Protein Bakery Brownie',
              '$1 Chocolate',
              'Vegetarian Fiesta Bowl',
              'Unchicken/Beyond Mission Burrito Wrap',
              'Cali Bowl',
              'Caesar Blt Chicken Wrap',
              'Spicy Chicken Panini',
              'Southewestern Fiesta Wrap',
              'Bbq Chicken Panini',
              'Pineapple Creamsicle',
              'Pineapple Creamsicle 2',
              'Apple Tart',
              'Sweet Street Brownie',
              'Harvest Signature Bowl',
              'Mediterranean',
              'Fresh Fitness Blast',
              'Turkey Pesto Panini',
              'P&J Panini',
              'Muffin',
              'Breakfast Wrap',
              'Turkey Bacon Avocado Wrap',
              'Turkey Bacon Avocado Panini',
              'Breakfast Panini',
              "Kid'S Combo With A Smoothie",
              'Fresh Fit Blast',
              'Cranberry Turkey Panini',
              'English Egg Muffin',
              
              'Energy Bites',
              'Soup Of The Day',
              'Sweets Un Sweetened',
              'Vegetarian $ Combo Meal',
              'Fresh Panini Platter', #assuming panini vegetarian
              'City Mandys Cookies', #assuming cookies are vegetarian
              'Pair A Wrap',
              'City Mandys Cream Pie', #assumption
              'Pair A Panini',
              'The Marshall',
              'Moroccan Glow Bowl',
              'Vegetarian Enjoy Life Lentil Chips',#checked the brand Enjoy Life
              'Sweet Bread',
              'The Lenny',
              'Protein Bakery Cookies',
              'Early Bird Panini',
              "Mandy'S Bakery"
              ]
vegetarian.extend(vegetarian_items)

# Is meat
meat = ['Chicken Caesar Salad',
        "Meat Beyond Burger",
        "Meat $ Combo Meal",
        "Mission Burrito Wrap",
        "Early Bird Wrap",
        "Turkey Bacon Avocado Panini",
        "Breakfast Panini",
        "Caesar Blt Chicken Wrap",
        "Beach Salad",
        "Mission Burrito Wrap",
        "Albacore Melt Panini",
        "Chicken $2 Combo",
        "Meat $ Combo Meal",
        "Signature Bowls", #assuming beach bowl
        "$9.99 Combo Meal",
        "Protein Bowl Box",
        "Protein Bites",
        "City Of Cape Protein Bowl", #assumption
        "City Salad Protein", #assumption
        "Big Salad Bowls",
        "Big Wrap Platter", #their wraps have cheese and meat
        "Box Lunch",
        "Fresh Soup",
        "Meat Salad Bowl Chicken/Turkey"
        ]
meat.extend(meat_items)

#very long list : Protein Bowl, Byo Protein/Salad Bowl

non_alcoholic_drinks = ["Iced Tea:  Organic Black Tea",
                        "Wheatgrass Shot",
                        "Green Lemonade",
                        "Ginger Shot",
                        "Kombucha",
                        "Specialty Iced Tea",
                        "The Buzz",
                        "16Oz Blue Spirulina",
                        "Green Lemonade Squeeze",
                        "Super Detox",
                        "Iced Tea Special  Tea",
                        "Mean Machine",
                        "Cyo (Create Your Own) Squeeze",
                        "Grab N Go"
                        ]

alcoholic_drinks = []

merch = []

rare =[]

unknown = ["Sweet Street Bar", "Big Panini Platter"
           "Loyalty Up Grade",
           "City Salad",
           "City 1/2 Wrap",
           "Heat And Serve",
           "Krispy"
           ]

items_to_remove = ["Gift Card", 
                   "Catering Utensils Per Person",
                   "Egift Card"
                   ]

In [ ]:
df_cleaned = clean_and_relabel_restaurant(
    df_uncleaned, 
    modification_name_changes, 
    vegan, 
    vegetarian, 
    meat, 
    [], # half vegan (items with no modifications but are known to be vegan half the time)
    alcoholic_drinks, 
    non_alcoholic_drinks, 
    merch, 
    rare, 
    unknown, 
    items_to_remove
    )

In [ ]:
df_cleaned.query('vegan')['item_name'].value_counts()

In [ ]:
df_cleaned.query('~vegan and vegetarian')['item_name'].value_counts()

In [ ]:
df_cleaned.query('~vegetarian')['item_name'].value_counts()

In [ ]:
print(
    food_df
    .query('is_plant_based == "Yes"')
    ['item_name'].value_counts().to_string())

In [ ]:
from tools.coverage_functions import plot_time_series
plot_time_series('JHDN7CF1C03X5', 
                 df_uncleaned.query('item_name.str.contains("Beyond Sausage")'), 
                 before_after_details_true, 
                 freq='W', 
                 subset=False)
plt.show()

In [ ]:
from tools.coverage_functions import plot_time_series
plot_time_series('JHDN7CF1C03X5', 
                 df_uncleaned, 
                 before_after_details_true, 
                 freq='W', 
                 subset=False)
plt.show()

In [ ]:
from tools.coverage_functions import plot_time_series
plot_time_series('JHDN7CF1C03X5', 
                 df_uncleaned.query('item_name.str.contains("Unchicken")'), 
                 before_after_details_true, 
                 freq='W', 
                 subset=False)
plt.show()

In [ ]:
# Visualizing with gaps for inactive weeks
introduction_fig, ax = plt.subplots(figsize=(14, 8))

# Index into the promotional items for this restaurant
promo_datetime = before_after_details_true.loc[loc_id,'cross_over_date'].tz_convert('UTC')

top_n = 30

unique_dishes = (df_uncleaned
                 ['item_name']
                 .value_counts()
                 .to_frame(name='c')
                 [:top_n]
                 .index[::-1]
                 )

legend_handles = []

for dish in unique_dishes:
    
    dish_df = df_uncleaned.query('item_name == @dish')
    
    vmin = dish_df['unit_price'].min()
    vmax = dish_df['unit_price'].max()
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cmap = cm.ScalarMappable(norm=norm, cmap='magma')
    
    weekly_quantities = (dish_df
                         .resample('W')
                         .agg({'item_quantity': 'sum', 'unit_price': 'mean'})
                         .query('0 < item_quantity')
                         .assign(week = lambda df: df.index.tz_localize(None).to_period('W'))
                         .set_index('week')
                         )
    
    # For every active week
    for week, row  in weekly_quantities.iterrows():
        
        weekly_quantity = row['item_quantity']
        dot_size = weekly_quantity/10 + 3
        weekly_price = row['unit_price']
        color = cmap.to_rgba(weekly_price)

        # Place a blue dot
        ax.hlines(y=dish, xmin=week.start_time, xmax=week.end_time, colors=color, lw=dot_size, label=loc_id)
        
    ax.text(x=food_df.index[-1] + pd.DateOffset(100), y=dish, s=f'${vmin/100:.2f}-${vmax/100:.2f}', verticalalignment='center', horizontalalignment='left', fontsize='x-small', color='gray')
    
    # Create a custom legend entry for this dish
    #color_patch_min = mpatches.Patch(color=cmap.to_rgba(vmin), label=f'{dish} Min: ${vmin/100:.2f}')
    #color_patch_max = mpatches.Patch(color=cmap.to_rgba(vmax), label=f'{dish} Max: ${vmax/100:.2f}')
    #legend_handles.extend([color_patch_min, color_patch_max])

# Place a red circle for the promotional item
ax.plot(promo_datetime, loc_id, 'ro', alpha=0.5)

# Plot
ax.set_title(f'Weekly Sales of Top {top_n} Dishes for {loc_id}')
ax.set_xlabel('Date')
ax.set_ylabel('Dish')
#ax.legend(handles=legend_handles, title="Price Range per Dish", fontsize='small', loc='upper left', bbox_to_anchor=(1, 1))

# Figure
introduction_fig.tight_layout(rect=[0, 0, 0.85, 1])

plt.show()

In [ ]:
# df.query('~vegetarian')['unit_price'].mean()
# df.query('is_plant_based == "Yes"')['item_name'].nunique() / df['item_name'].nunique()
